**Note:** This notebook stays close to the original project. Only the local database path and small variable-name issues are fixed. The invoice flag is rule-based exploratory labeling, so it should not be treated as an independent ML ground truth.


In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
from pathlib import Path
import sqlite3
db_path = Path("../inventory.db") if Path("../inventory.db").exists() else Path("inventory.db")
conn = sqlite3.connect(db_path)
tables = pd.read_sql_query("select name from sqlite_master where type='table'", conn)


In [ ]:
for table in tables['name']:
    print("Table Name",table)
    df=pd.read_sql_query(f"select * from {table} limit 5",conn)
    display(df)

In [ ]:
purchases_agg=pd.read_sql_query("""select PONumber,count(distinct Brand) as total_brand
,sum(Quantity) as total_quantity,sum(Dollars) as total_dollar,avg(julianday (ReceivingDate) - julianday(PODate))
as avg_receving_dely from purchases group by PONumber""",conn)

In [ ]:
pd.read_sql_query(""" select PONumber, Quantity as invoice_quantity,Dollars as invoice_dollars
,Freight,julianday(InvoiceDate) - julianday(PODate) as days_po_to_invoice,
julianday(PayDate)-julianday(InvoiceDate) as days_to_pay from vendor_invoice""",conn)

In [ ]:
df=pd.read_sql_query("""with purchase_agg as(select PONumber,count(distinct Brand) as total_brand
,sum(Quantity) as total_quantity,sum(Dollars) as total_dollar,avg(julianday (ReceivingDate) - julianday(PODate))
as avg_receving_dely from purchases group by PONumber)

select v.PONumber, Quantity as invoice_quantity,Dollars as invoice_dollars
,Freight,julianday(InvoiceDate) - julianday(PODate) as days_po_to_invoice,
julianday(PayDate)-julianday(InvoiceDate) as days_to_pay,total_dollar,
total_brand,total_quantity,avg_receving_dely from vendor_invoice v left join purchase_agg p on p.PONumber=v.PONumber
""",conn)


In [ ]:
df.isnull().sum()

In [ ]:
df

In [ ]:
def invoice_risk(row):
    if(abs(row['invoice_dollars']-row['total_dollar'])>5):
        return 1

    if(row['avg_receving_dely']>10):
        return 1 

    return 0

df['flagged_invoice']=df.apply(invoice_risk,axis=1)
df['flagged_invoice'].value_counts()

In [ ]:
df['flagged_invoice'].value_counts().plot(kind='bar')

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.corr(),annot=True)

In [ ]:
flagged=df[df['flagged_invoice']==1]
normal=df[df['flagged_invoice']==0]

In [ ]:
significant_features = []
non_significant_features = []
results = []

In [ ]:
metrics = [
    'invoice_quantity',
    'invoice_dollars',
    'Freight',
    'days_po_to_invoice',
    'days_to_pay',
    'total_brand',
    'total_quantity',
    'total_dollar',
    'avg_receving_dely'
]

In [ ]:
from scipy.stats import ttest_ind

for metric in metrics:
    flagged_mean=flagged[metric].mean()
    normal_mean=normal[metric].mean()

    t_stat,p_value=ttest_ind(flagged[metric].dropna(), normal[metric].dropna(), equal_var=False)

    if p_value<0.05:
        significant_features.append(metric)
        results.append({
            "metrics":metric,
            "flagged_mean":flagged_mean.round(2),
            "normal_mean":normal_mean.round(2),
            "p_value":p_value.round(3)
        })
    else:
        non_significant_features.append(metric)

In [ ]:
df.columns

In [ ]:
non_significant_features

In [ ]:
significant_features

In [ ]:
results

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
X = df[['invoice_quantity',
        'invoice_dollars',
        'Freight',
        'days_po_to_invoice',
        'total_quantity',
        'total_dollar',
        'avg_receving_dely']]

y = df['flagged_invoice']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
X.describe().round()

In [ ]:
from sklearn.metrics import classification_report

def detect(model, x_test, y_test, model_name):
    pred = model.predict(x_test)

    print(f"\nModel: {model_name}")
    print(classification_report(y_test, pred))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
x_trained=scaler.fit_transform(X_train)
x_tested = scaler.transform(X_test)

In [ ]:
model1 = LogisticRegression(random_state=42)
model1.fit(x_trained, y_train)

model2 = DecisionTreeClassifier(random_state=42)
model2.fit(x_trained, y_train)

model3 = RandomForestClassifier(random_state=42)
model3.fit(x_trained, y_train)

In [ ]:
detect(model1, x_tested, y_test, "Logistic Regression")
detect(model2, x_tested, y_test, "Decision Tree")
detect(model3, x_tested, y_test, "Random Forest")

In [ ]:
feature_importance=pd.DataFrame({
    "Feature":X_train.columns,
    "importance":model3.feature_importances_
}).sort_values(by="importance",ascending=False)

In [ ]:
feature_importance

In [ ]:
X = df[['invoice_quantity',
        'avg_receving_dely',
        'Freight',
        'total_quantity',
        'total_dollar',]]

y = df['flagged_invoice']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
x_trained=scaler.fit_transform(X_train)
x_tested = scaler.transform(X_test)

In [ ]:
model3 = RandomForestClassifier(random_state=42)
model3.fit(x_trained, y_train)
detect(model3, x_tested, y_test, "Random Forest")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42,n_jobs=-1)
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 4, 5, 6],
    "min_samples_split": [2, 3, 5],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}
scorer = make_scorer(f1_score)
grid_search = GridSearchCV(estimator=rf,param_grid=param_grid,scoring=scorer,cv=5,verbose=2,n_jobs=-1)
grid_search.fit(x_trained, y_train)
print("Best Parameters:")
print(grid_search.best_params_)
print("\nBest F1 Score:")
print(grid_search.best_score_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_tested)
test_f1 = f1_score(y_test, y_pred)
print("\nTest F1 Score:")
print(test_f1)